# LLM Judge with GPT-OSS 20B on Google Colab

This notebook acts as an **LLM Judge** to evaluate and compare tutorials generated by the DeepAgent pipeline.
It uses the [openai/gpt-oss-20b](https://hf.co/openai/gpt-oss-20b) model running locally on a free Colab GPU (T4).

## 1. Setup Environment

In [ ]:
!pip install -q --upgrade torch transformers triton==3.4 kernels litellm smolagents loguru

In [ ]:
!pip uninstall -q torchvision torchaudio -y

**⚠️ IMPORTANT:** restart your Colab runtime session now (Runtime > Restart session).

## 2. Load Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "openai/gpt-oss-20b"

print(f"Loading {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="cuda",
)
print("Model loaded successfully!")

## 3. Define Logic

In [ ]:
from smolagents import LLMModel
from typing import List, Dict, Any, Optional
import json
import re
from loguru import logger
import sys

# Configure logging to stdout
logger.remove()
logger.add(sys.stdout, format="<green>{time:HH:mm:ss}</green> | <level>{message}</level>")

class LocalHuggingFaceModel(LLMModel):
    """Wrapper for local HF model to work with smolagents/litellm interface."""
    def __init__(self, model, tokenizer):
        super().__init__()
        self.model = model
        self.tokenizer = tokenizer

    def __call__(
        self,
        messages: List[Dict[str, str]],
        stop_sequences: Optional[List[str]] = None,
        grammar: Optional[str] = None,
        flatten_messages_as_text: bool = False,
        **kwargs,
    ) -> Any:
        
        # Apply chat template
        inputs = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        ).to(self.model.device)

        # Generate
        generated_ids = self.model.generate(
            **inputs,
            max_new_tokens=kwargs.get("max_tokens", 2048),
            do_sample=True,
            temperature=0.2,
        )
        
        # Decode response (skip input prompt)
        input_len = inputs["input_ids"].shape[-1]
        output_ids = generated_ids[0][input_len:]
        response_text = self.tokenizer.decode(output_ids, skip_special_tokens=True)
        
        return response_text

# Initialize the wrapper
judge_model = LocalHuggingFaceModel(model, tokenizer)


In [ ]:
COMPARE_PROMPT = """You are an expert technical documentation reviewer.
Your task is to compare two versions of a tutorial for the same topic and decide which one is better.

## Codebase Context
{codebase_context}

## Tutorial Version A
{content_a}

## Tutorial Version B
{content_b}

## Instructions
1. Read the Codebase Context to understand the topic.
2. Read both tutorials.
3. Compare them based on:
    - **Accuracy**: Does it match the code?
    - **Clarity**: Is it easy to understand?
    - **Completeness**: Does it solve the problem?
    - **Code Quality**: Are examples robust?

## Response Format
Return ONLY valid JSON (no markdown fences):
{{
    "winner": "A" or "B" or "Tie",
    "rationale": "Explanation...",
    "scores": {{
        "A": <1-5>,
        "B": <1-5>
    }}
}}
"""

In [ ]:
def _truncate(text: str, max_chars: int) -> str:
    """Truncate text to fit context window."""
    if len(text) <= max_chars:
        return text
    half = (max_chars - 100) // 2
    return text[:half] + "\n\n... [truncated] ...\n\n" + text[-half:]

def evaluate_pair(model, file_name, codebase_context, content_a, content_b):
    """Compare two tutorials using the local model."""
    
    prompt = COMPARE_PROMPT.format(
        codebase_context=codebase_context,
        content_a=_truncate(content_a, 4000),
        content_b=_truncate(content_b, 4000)
    )
    
    logger.info(f"Evaluating {file_name}...")
    try:
        response = model(
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1000
        )
        
        # Parse JSON
        content = str(response).strip()
        content = re.sub(r"^```(?:json)?\s*", "", content, flags=re.IGNORECASE).strip()
        content = re.sub(r"\s*```$", "", content).strip()
        
        # Fix common JSON errors
        content = re.sub(r",\s*\}", "}", content)
        
        try:
            data = json.loads(content)
            data["file_name"] = file_name
            return data
        except json.JSONDecodeError:
            # Regex fallback
            match = re.search(r"(\{.*\})", content, flags=re.DOTALL)
            if match:
                data = json.loads(match.group(1))
                data["file_name"] = file_name
                return data
            else:
                return {"file_name": file_name, "winner": "Error", "rationale": f"Invalid JSON: {content[:100]}..."}
                
    except Exception as e:
        logger.error(f"Error evaluating {file_name}: {e}")
        return {"file_name": file_name, "winner": "Error", "rationale": str(e)}

## 4. Run Evaluation

1. Mount Google Drive.
2. Configure paths to your tutorial artifacts on Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob
from pathlib import Path

# --- CONFIGURATION ---
# Update these paths to match your Drive structure
BASELINE_PATH = Path("/content/drive/MyDrive/data/baseline/tutorials")
DEEP_PATH = Path("/content/drive/MyDrive/data/deep_agent/tutorials")
CODEBASE_CONTEXT_FILE = Path("/content/drive/MyDrive/data/rag_vector_store/codebase_context.txt") # Optional: predefined context

# Or just a hardcoded context for testing
CODEBASE_CONTEXT = """ 
# Project: Instructor (Python)
Instructor is a Python library for structured outputs from LLMs.
It patches OpenAI/Anthropic/Gemini clients to return Pydantic models.
""" 
# ---------------------

baseline_files = set(f.name for f in BASELINE_PATH.glob("*.md"))
deep_files = set(f.name for f in DEEP_PATH.glob("*.md"))
common = sorted(list(baseline_files.intersection(deep_files)))

print(f"Found {len(common)} tutorials to compare: {common}")

results = []
if common:
    for fname in common:
        path_a = BASELINE_PATH / fname
        path_b = DEEP_PATH / fname
        
        res = evaluate_pair(
            judge_model,
            fname,
            CODEBASE_CONTEXT,
            path_a.read_text(),
            path_b.read_text()
        )
        results.append(res)
        print(f"[{fname}] Winner: {res.get('winner')} | Score A: {res.get('scores', {}).get('A')} | Score B: {res.get('scores', {}).get('B')}")

    # Summary
    wins = {"A": 0, "B": 0, "Tie": 0, "Error": 0}
    for r in results:
        w = r.get("winner", "Error")
        wins[w] = wins.get(w, 0) + 1
        
    print("\n=== SUMMARY ===")
    print(f"Baseline (A): {wins['A']}")
    print(f"DeepAgent (B): {wins['B']}")
    print(f"Ties: {wins['Tie']}")
else:
    print("No common files found. Check your paths!")